# CDT-III 2-Stage VCE: CTCF Enrichment + Hi-C Validation
## Lightweight notebook — no attention map storage, minimal memory

**Purpose**: CTCF enrichment analysis (28 genes) + Hi-C 3D contact validation.
Separated from NB4 to avoid OOM from large attention tensors.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
from scipy import stats as scipy_stats
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import gzip
import io
import gc
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2")

CELLLEVEL_WITH_ADT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
BEST_MODEL_PATH = OUTPUT_BASE / "cdt_2stage_vce_best.pt"

FIGURES_DIR = OUTPUT_BASE / "attention_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENFORMER_BIN_SIZE = 128
ENFORMER_N_BINS = 896
ENFORMER_CENTER_BIN = ENFORMER_N_BINS // 2

for name, path in [("Cell-level + ADT", CELLLEVEL_WITH_ADT_PATH),
                    ("TSS Enformer", TSS_ENFORMER_PATH),
                    ("Best model", BEST_MODEL_PATH)]:
    print(f"  {'\u2713' if path.exists() else '\u2717'} {name}: {path}")

In [ ]:
# ============================================================
# CDT-III 2-Stage VCE Model Definition
# ============================================================

@dataclass
class CDT2StageVCEConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    n_proteins: int = 189
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    protein_dropout: float = 0.5
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.feature_embedding = nn.Embedding(n_features, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout))
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

    def forward(self, expression):
        B = expression.size(0)
        feat_ids = torch.arange(self.n_features, device=expression.device)
        feat_emb = self.feature_embedding(feat_ids).unsqueeze(0).expand(B, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        return self.combine(torch.cat([feat_emb, expr_emb], dim=-1))


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        B, S, _ = x.shape
        Q = self.q_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, return_attn=False):
        B, QL, _ = query.shape
        KL = key_value.shape[1]
        Q = self.q_proj(query).view(B, QL, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, QL, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj)
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj)
        return self.fusion(torch.cat([dna_pooled, rna_pooled], dim=-1))


class VirtualCellEmbedderProtein(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_q_proj = nn.Linear(d_model, d_model)
        self.protein_k_proj = nn.Linear(d_model, d_model)
        self.protein_v_proj = nn.Linear(d_model, d_model)
        self.protein_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, cell_emb_rna, protein_encoded):
        protein_pooled = self._attention_pool(
            self.protein_query, protein_encoded,
            self.protein_q_proj, self.protein_k_proj,
            self.protein_v_proj, self.protein_out_proj)
        return self.fusion(torch.cat([cell_emb_rna, protein_pooled], dim=-1))


class CDTTrimodal2StageModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDT2StageVCEConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce_t = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.rna_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.dropout), nn.Linear(config.hidden_dim, config.n_genes))
        self.protein_encoder = RawExpressionEncoder(
            config.n_proteins, config.hidden_dim, config.protein_dropout)
        self.protein_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.protein_dropout)
            for _ in range(config.protein_self_attn_layers)])
        self.rna_to_protein = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.protein_dropout)
        self.vce_p = VirtualCellEmbedderProtein(config.hidden_dim, config.protein_dropout)
        self.protein_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.protein_dropout),
            nn.Linear(config.hidden_dim, config.n_proteins))

    def forward(self, dna_emb, rna_expr, protein_expr, return_attention=False):
        attn_maps = {} if return_attention else None
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        protein = self.protein_encoder(protein_expr)
        for i, layer in enumerate(self.dna_self_attn_layers):
            if return_attention:
                dna, attn_w = layer(dna, return_attn=True)
                attn_maps[f'dna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                dna = layer(dna)
        for i, layer in enumerate(self.rna_self_attn_layers):
            if return_attention:
                rna, attn_w = layer(rna, return_attn=True)
                attn_maps[f'rna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                rna = layer(rna)
        for i, layer in enumerate(self.protein_self_attn_layers):
            if return_attention:
                protein, attn_w = layer(protein, return_attn=True)
                attn_maps[f'protein_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                protein = layer(protein)
        if return_attention:
            rna, attn_w = self.dna_to_rna(query=rna, key_value=dna, return_attn=True)
            attn_maps['dna_to_rna_cross'] = attn_w.detach().cpu()
        else:
            rna = self.dna_to_rna(query=rna, key_value=dna)
        if return_attention:
            protein, attn_w = self.rna_to_protein(query=protein, key_value=rna, return_attn=True)
            attn_maps['rna_to_protein_cross'] = attn_w.detach().cpu()
        else:
            protein = self.rna_to_protein(query=protein, key_value=rna)
        cell_emb_rna = self.vce_t(dna, rna)
        rna_pred = self.rna_task_layer(cell_emb_rna)
        cell_emb_protein = self.vce_p(cell_emb_rna, protein)
        protein_pred = self.protein_task_layer(cell_emb_protein)
        if return_attention:
            return rna_pred, protein_pred, attn_maps
        return rna_pred, protein_pred

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("CDTTrimodal2StageModel defined.")

In [ ]:
# ============================================================
# Load Data
# ============================================================
print("Loading integrated RNA + ADT data...")
with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    rna_log2fc = f['log2fc'][:]
    cell_expr = f['cell_expr'][:]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g for g in f['val_genes'][:]]
    protein_dsb = f['protein_dsb'][:]
    protein_names = [g.decode() if isinstance(g, bytes) else g for g in f['protein_names'][:]]
    n_cells = f.attrs['n_cells']
    n_genes = f.attrs['n_genes']
    n_proteins = f.attrs['n_proteins']

    if 'protein_matched_mask' in f:
        protein_matched_mask = f['protein_matched_mask'][:]
    else:
        print("Warning: 'protein_matched_mask' not found. Creating all-True mask.")
        protein_matched_mask = np.ones(n_cells, dtype=bool)

print(f"RNA: {n_cells} cells, {n_genes} genes")
print(f"Protein: {n_proteins} proteins")
print(f"Cells with protein: {protein_matched_mask.sum()}/{n_cells}")
print(f"Val genes: {val_genes}")

# Enformer
print("\nLoading Enformer embeddings...")
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    enformer_emb = f['embeddings'][:]
    enformer_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
    tss_coords = f['tss_coords'][:]
    tss_chroms = [c.decode() if isinstance(c, bytes) else c for c in f['chroms'][:]]
    tss_strands = [s.decode() if isinstance(s, bytes) else s for s in f['strands'][:]]
print(f"Enformer: {enformer_emb.shape}, TSS: {len(tss_coords)} genes")

target_to_enformer = {}
for i, gene in enumerate(target_gene_names):
    if gene in enformer_genes:
        target_to_enformer[i] = enformer_genes.index(gene)
print(f"TSS matched: {len(target_to_enformer)}/{len(target_gene_names)}")

gene_to_cdt_idx = {g: i for i, g in enumerate(cdt_genes)}
gene_name_to_target_idx = {n: i for i, n in enumerate(target_gene_names)}
gene_to_enformer = {g: i for i, g in enumerate(enformer_genes)}

In [ ]:
import os
output_dir = "/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/"
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f} ({size/1e6:.1f} MB)")
else:
    print("Directory not found!")
    parent = "/content/drive/MyDrive/cdt_outputs/"
    if os.path.exists(parent):
        print("Available dirs:")
        for d in sorted(os.listdir(parent)):
            print(f"  {d}")

In [ ]:
# ============================================================
# Load Best Model Checkpoint
# ============================================================
print("Loading best model checkpoint...")
config = CDT2StageVCEConfig(n_genes=len(cdt_genes), n_proteins=len(protein_names))
model = CDTTrimodal2StageModel(config).to(device)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded from checkpoint with keys: {list(ckpt.keys())}")
    if 'val_rna_r' in ckpt:
        print(f"  Val RNA r: {ckpt['val_rna_r']:.4f}")
    if 'val_protein_r' in ckpt:
        print(f"  Val Protein r: {ckpt['val_protein_r']:.4f}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state dict")

del ckpt
gc.collect()
model.eval()
print(f"Parameters: {model.get_num_params():,}")

In [ ]:
# ============================================================
# CTCF ChIP-seq Download (ENCODE K562)
# ============================================================

CTCF_PEAKS_URL = "https://www.encodeproject.org/files/ENCFF796WRU/@@download/ENCFF796WRU.bed.gz"
CTCF_LOCAL = OUTPUT_BASE / "k562_ctcf_peaks.bed"

if not CTCF_LOCAL.exists():
    print("Downloading K562 CTCF peaks from ENCODE...")
    resp = requests.get(CTCF_PEAKS_URL)
    with gzip.open(io.BytesIO(resp.content), "rt") as gz:
        content = gz.read()
    with open(CTCF_LOCAL, "w") as f:
        f.write(content)
    print(f"  Saved to {CTCF_LOCAL}")
else:
    print(f"  Already downloaded: {CTCF_LOCAL}")

ctcf_peaks = {}
with open(CTCF_LOCAL) as f:
    for line in f:
        parts = line.strip().split("\t")
        chrom = parts[0]
        start, end = int(parts[1]), int(parts[2])
        signal = float(parts[6]) if len(parts) > 6 else 1.0
        ctcf_peaks.setdefault(chrom, []).append((start, end, signal))

for ch in ctcf_peaks:
    ctcf_peaks[ch].sort()

total_peaks = sum(len(v) for v in ctcf_peaks.values())
print(f"CTCF peaks: {total_peaks} across {len(ctcf_peaks)} chroms")

In [ ]:
# ============================================================
# Cross-Attention Extraction (28 genes) + CTCF Enrichment
# Memory-efficient: only keeps [896] profile per gene, not full attention
# ============================================================

def find_ctcf_in_window(chrom, tss_coord):
    ws = tss_coord - ENFORMER_CENTER_BIN * ENFORMER_BIN_SIZE
    we = tss_coord + ENFORMER_CENTER_BIN * ENFORMER_BIN_SIZE
    peaks = ctcf_peaks.get(chrom, [])
    return [(s, e, sig) for s, e, sig in peaks if s < we and e > ws], ws, we

def ctcf_to_bins(ctcf_list, tss_coord):
    bins = []
    for start, end, signal in ctcf_list:
        center = (start + end) // 2
        b = (center - tss_coord) // ENFORMER_BIN_SIZE + ENFORMER_CENTER_BIN
        if 0 <= b < ENFORMER_N_BINS:
            bins.append((b, signal, start, end))
    return bins

def extract_cross_attn_profile(model, dna_emb_np, rna_expr_np, prot_expr_np, device):
    """Extract DNA->RNA cross-attention profile [896] without storing full attention."""
    dna_t = torch.from_numpy(dna_emb_np[np.newaxis].astype(np.float32)).to(device)
    rna_t = torch.from_numpy(rna_expr_np[np.newaxis].astype(np.float32)).to(device)
    prot_t = torch.from_numpy(prot_expr_np[np.newaxis].astype(np.float32)).to(device)

    with torch.no_grad():
        dna = model.dna_projector(dna_t)
        rna = model.rna_encoder(rna_t)
        for layer in model.dna_self_attn_layers:
            dna = layer(dna)
        for layer in model.rna_self_attn_layers:
            rna = layer(rna)

        ca = model.dna_to_rna
        B, QL = rna.shape[0], rna.shape[1]
        KL = dna.shape[1]
        Q = ca.q_proj(rna).view(B, QL, ca.nhead, ca.head_dim).transpose(1, 2)
        K = ca.k_proj(dna).view(B, KL, ca.nhead, ca.head_dim).transpose(1, 2)
        attn_logits = torch.matmul(Q, K.transpose(-2, -1)) / (ca.head_dim ** 0.5)
        attn_weights = torch.softmax(attn_logits, dim=-1)
        # Average over heads and query genes -> [896]
        profile = attn_weights[0].mean(dim=0).mean(dim=0).cpu().numpy()

    del dna_t, rna_t, prot_t, dna, rna, Q, K, attn_logits, attn_weights
    return profile


gene_attn_profiles = {}
print("Extracting cross-attention for 28 genes...")

for gene in tqdm(enformer_genes):
    enf_idx = gene_to_enformer[gene]
    dna_emb_g = enformer_emb[enf_idx]

    g_target = gene_name_to_target_idx.get(gene)
    if g_target is not None:
        g_cells = [i for i in range(n_cells)
                   if target_gene_idx[i] == g_target
                   and g_target in target_to_enformer
                   and protein_matched_mask[i]]
    else:
        g_cells = []

    if len(g_cells) >= 5:
        n_used = min(20, len(g_cells))
        profiles = []
        for ci in g_cells[:n_used]:
            p = extract_cross_attn_profile(
                model, dna_emb_g, cell_expr[ci], protein_dsb[ci], device)
            profiles.append(p)
        gene_attn_profiles[gene] = np.mean(profiles, axis=0)
    else:
        gene_attn_profiles[gene] = extract_cross_attn_profile(
            model, dna_emb_g, ntc_mean_expr, np.zeros(n_proteins, dtype=np.float32), device)

    gc.collect()

# CTCF enrichment
ctcf_results = []
for gene in enformer_genes:
    if gene not in gene_attn_profiles:
        continue
    enf_idx = gene_to_enformer[gene]
    chrom = tss_chroms[enf_idx]
    tss = tss_coords[enf_idx]
    attn = gene_attn_profiles[gene]

    ctcf_in_window, _, _ = find_ctcf_in_window(chrom, tss)
    ctcf_bins = ctcf_to_bins(ctcf_in_window, tss)

    n_top = int(ENFORMER_N_BINS * 0.1)
    top_bins = set(np.argsort(attn)[-n_top:])

    ctcf_in_top = sum(1 for b, _, _, _ in ctcf_bins if b in top_bins)
    n_ctcf = len(ctcf_bins)
    expected = n_ctcf * 0.1 if n_ctcf > 0 else 0
    enrichment = ctcf_in_top / expected if expected > 0 else float('nan')

    ctcf_results.append({
        'gene': gene, 'chrom': chrom, 'tss': tss,
        'n_ctcf': n_ctcf, 'ctcf_in_top': ctcf_in_top,
        'enrichment': enrichment, 'attn': attn, 'ctcf_bins': ctcf_bins,
    })

ctcf_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('attn', 'ctcf_bins')}
                         for r in ctcf_results])
print(ctcf_df[['gene', 'n_ctcf', 'ctcf_in_top', 'enrichment']].to_string(index=False))
print(f"\nMean enrichment: {ctcf_df['enrichment'].mean():.2f}x")
print(f"Genes >2x: {(ctcf_df['enrichment'] > 2).sum()}/{len(ctcf_df)}")

In [ ]:
# ============================================================
# FREE ALL MEMORY before Hi-C
# ============================================================

try:
    model.cpu()
    del model
except:
    pass

for var in ['enformer_emb', 'cell_expr', 'protein_dsb', 'rna_log2fc',
            'ntc_mean_expr', 'protein_matched_mask', 'target_gene_idx',
            'protein_names', 'cdt_genes', 'target_gene_names',
            'train_genes', 'val_genes', 'gene_attn_profiles']:
    if var in dir():
        try:
            exec(f'del {var}')
        except:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"ctcf_results: {len(ctcf_results)} genes retained")
print("Memory freed. Ready for Hi-C.")

In [ ]:
# ============================================================
# Hi-C Three-Layer Validation (v4: remote straw, small queries)
# ============================================================
# Instead of downloading the full .hic file (crashes Colab),
# use hicstraw with SMALL targeted queries (+/-100kb per gene)
# with memory freed and 1s delay between queries

!pip install -q hic-straw
import hicstraw
import time

ENFORMER_BIN_SIZE = 128
ENFORMER_N_BINS = 896
ENFORMER_CENTER_BIN = ENFORMER_N_BINS // 2

HIC_URL = "https://4dn-open-data-public.s3.amazonaws.com/fourfront-webprod/wfoutput/7386f953-8f9a-4bd4-9296-5f4a8d2f8fd6/4DNFI18UHVRO.hic"
HIC_RESOLUTION = 5000

# Test with a TINY query first
hic_available = False
try:
    test = hicstraw.straw("observed", "KR", HIC_URL,
                          "1:50000000:50005000", "1:50000000:50005000",
                          "BP", HIC_RESOLUTION)
    print(f"hicstraw remote OK: {len(test)} records")
    hic_available = True
    del test
    gc.collect()
except Exception as e:
    print(f"hicstraw failed: {e}")

if hic_available:
    hic_results = []
    n_errors = 0

    for gi, r in enumerate(ctcf_results):
        gene = r['gene']
        chrom = r['chrom']
        tss = r['tss']
        attn = r['attn']
        ctcf_bins = r['ctcf_bins']
        cn = chrom.replace("chr", "")

        if len(ctcf_bins) < 3:
            hic_results.append({'gene': gene, 'hic_ratio': np.nan})
            print(f"  [{gi+1}/{len(ctcf_results)}] {gene}: <3 CTCF, skip")
            continue

        # Median-split CTCF by attention weight
        ctcf_with_attn = [(b, attn[b]) for b, _, _, _ in ctcf_bins]
        ctcf_with_attn.sort(key=lambda x: x[1], reverse=True)
        mid = max(1, len(ctcf_with_attn) // 2)
        high_attn_ctcf = [x[0] for x in ctcf_with_attn[:mid]]
        low_attn_ctcf = [x[0] for x in ctcf_with_attn[mid:]]

        if not high_attn_ctcf or not low_attn_ctcf:
            hic_results.append({'gene': gene, 'hic_ratio': np.nan})
            continue

        # Small query: TSS +/- 100kb only
        qs = max(0, tss - 100000)
        qe = tss + 100000
        tss_bin = (tss // HIC_RESOLUTION) * HIC_RESOLUTION
        ws = tss - ENFORMER_CENTER_BIN * ENFORMER_BIN_SIZE

        try:
            result = hicstraw.straw(
                "observed", "KR", HIC_URL,
                f"{cn}:{tss_bin}:{tss_bin + HIC_RESOLUTION}",
                f"{cn}:{qs}:{qe}", "BP", HIC_RESOLUTION)

            contact_map = {}
            for rec in result:
                pos = rec.binX if rec.binX != tss_bin else rec.binY
                ebin = (pos - ws) // ENFORMER_BIN_SIZE
                if 0 <= ebin < ENFORMER_N_BINS:
                    contact_map[ebin] = max(contact_map.get(ebin, 0), rec.counts)

            ma = float(np.mean([contact_map.get(b, 0) for b in high_attn_ctcf]))
            mn = float(np.mean([contact_map.get(b, 0) for b in low_attn_ctcf]))
            ratio = ma / mn if mn > 0 else np.nan

            hic_results.append({
                'gene': gene, 'n_high': len(high_attn_ctcf), 'n_low': len(low_attn_ctcf),
                'high_hic': ma, 'low_hic': mn, 'hic_ratio': ratio,
                'n_contacts': len(contact_map)})

            print(f"  [{gi+1}/{len(ctcf_results)}] {gene}: ratio={ratio:.2f}x "
                  f"(high={ma:.1f}, low={mn:.1f}, contacts={len(contact_map)})")
            del result, contact_map

        except Exception as e:
            hic_results.append({'gene': gene, 'hic_ratio': np.nan})
            n_errors += 1
            print(f"  [{gi+1}/{len(ctcf_results)}] {gene}: error - {str(e)[:60]}")
            if n_errors >= 5:
                print(f"  Too many errors, stopping.")
                break

        gc.collect()
        time.sleep(1)

    hic_df = pd.DataFrame(hic_results)
    valid = hic_df.dropna(subset=['hic_ratio'])
    print(f"\nHi-C Summary (median-split):")
    print(f"  Valid: {len(valid)}/{len(hic_results)}, Errors: {n_errors}")
    if len(valid) > 0:
        print(f"  Mean ratio: {valid['hic_ratio'].mean():.2f}x")
        print(f"  Median: {valid['hic_ratio'].median():.2f}x")
        print(f"  Genes >1x: {(valid['hic_ratio'] > 1).sum()}/{len(valid)}")
    if len(valid) >= 3:
        from scipy import stats as scipy_stats
        t, p = scipy_stats.ttest_1samp(valid['hic_ratio'].dropna(), 1.0)
        print(f"  t-test vs 1.0: t={t:.3f}, p={p:.4f}")
else:
    hic_df = None
    print("Hi-C skipped.")

In [ ]:
# ============================================================
# Permutation Test (n=1000) for CTCF Enrichment
# ============================================================

n_perm = 1000
perm_enrichments = []
for _ in range(n_perm):
    total_obs, total_exp = 0, 0
    for r in ctcf_results:
        nc = len(r['ctcf_bins'])
        if nc == 0:
            continue
        rand_top = set(np.random.choice(ENFORMER_N_BINS, 90, replace=False))
        total_obs += sum(1 for b, _, _, _ in r['ctcf_bins'] if b in rand_top)
        total_exp += nc * 0.1
    perm_enrichments.append(total_obs / total_exp if total_exp > 0 else 1.0)

obs_mean = ctcf_df['enrichment'].mean()
perm_p = np.mean([e >= obs_mean for e in perm_enrichments])

print(f"Permutation Test (n={n_perm}):")
print(f"  Observed enrichment: {obs_mean:.2f}x")
print(f"  Null: {np.mean(perm_enrichments):.2f} +/- {np.std(perm_enrichments):.2f}")
print(f"  p-value: {perm_p:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(perm_enrichments, bins=50, color='gray', alpha=0.7, label='Null distribution')
ax.axvline(obs_mean, color='red', linewidth=2, linestyle='--',
           label=f'Observed: {obs_mean:.2f}x (p={perm_p:.4f})')
ax.set_xlabel('CTCF Enrichment (fold)')
ax.set_ylabel('Count')
ax.set_title('CTCF Enrichment: Permutation Test', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ctcf_permutation_test.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# All-Genes Cross-Attention + CTCF Overview
# ============================================================

n_plot = len(ctcf_results)
ncols = 4
nrows = (n_plot + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(24, 4 * nrows), dpi=150)
axes_flat = axes.flatten()
rel_kb = (np.arange(ENFORMER_N_BINS) - ENFORMER_CENTER_BIN) * ENFORMER_BIN_SIZE / 1000

for idx, r in enumerate(ctcf_results):
    ax = axes_flat[idx]
    gene = r['gene']
    attn = r['attn']
    ctcf_bins = r['ctcf_bins']
    n_top = int(ENFORMER_N_BINS * 0.1)
    top_bins = set(np.argsort(attn)[-n_top:])

    ax.fill_between(rel_kb, attn / attn.max(), alpha=0.3, color="#1976D2")
    ax.plot(rel_kb, attn / attn.max(), color="#1976D2", linewidth=0.8)
    for b, sig, _, _ in ctcf_bins:
        color = "#D32F2F" if b in top_bins else "#FFCDD2"
        ax.axvline(rel_kb[b], color=color, alpha=0.5, linewidth=0.8, linestyle="--")
    ax.axvline(0, color="black", linewidth=1.5, alpha=0.7)
    ax.set_title(f"{gene} ({len(ctcf_bins)} CTCF, {r['enrichment']:.1f}x)",
                 fontsize=9, fontweight="bold")
    ax.set_xlim(rel_kb[0], rel_kb[-1])
    ax.set_ylim(0, 1.1)
    ax.tick_params(labelsize=7)

for i in range(len(ctcf_results), len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.suptitle("CDT-III: Cross-Attention + CTCF Peaks (28 Morris Genes)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'all_genes_crossattn_ctcf.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Save Results
# ============================================================

ctcf_df.to_csv(OUTPUT_BASE / 'ctcf_enrichment_results.csv', index=False)
if hic_df is not None:
    hic_df.to_csv(OUTPUT_BASE / 'hic_three_layer_results.csv', index=False)

np.savez(OUTPUT_BASE / 'cross_attention_profiles.npz',
         genes=np.array(list(gene_attn_profiles.keys())),
         profiles=np.array(list(gene_attn_profiles.values())))

print(f"\n{'='*60}")
print("CDT-III CTCF / Hi-C Summary")
print(f"{'='*60}")
print(f"CTCF enrichment: {ctcf_df['enrichment'].mean():.2f}x (CDT-II: 6.6x)")
print(f"Permutation p-value: {perm_p:.4f}")
if hic_df is not None:
    valid = hic_df.dropna(subset=['hic_ratio'])
    if len(valid) > 0:
        print(f"Hi-C ratio: {valid['hic_ratio'].mean():.2f}x")
print(f"\nSaved to {OUTPUT_BASE}")
print("Done.")